# Secure Data Orchestration Framework for ICU Monitoring

This notebook evaluates a scalable ICU monitoring analytics pipeline.

Workflow:
1. Load ICU monitoring dataset
2. Perform preprocessing and risk labeling
3. Train Logistic Regression classifier
4. Evaluate model performance
5. Perform clustering
6. Measure execution latency

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [ ]:
df = pd.read_csv("../data/processed_monitoring.csv")
df.head()

## Risk Label Definition

Risk = 1 if:
- HeartRate > 100 OR
- SpO2 < 92

In [ ]:
df["Risk"] = np.where(
    (df["HeartRate"] > 100) | (df["SpO2"] < 92),
    1,
    0
)

df.head()

## Train-Test Split

In [ ]:
features = ["HeartRate", "SystolicBP", "DiastolicBP", "SpO2"]
X = df[features]
y = df["Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## Logistic Regression Model Training
Execution time is measured to simulate distributed processing latency.

In [ ]:
start_time = time.time()

model = LogisticRegression()
model.fit(X_train, y_train)

end_time = time.time()
execution_time = end_time - start_time

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print("Accuracy:", round(accuracy,3))
print("Precision:", round(precision,3))
print("Recall:", round(recall,3))
print("Execution Time:", round(execution_time,5), "seconds")

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## ROC Curve

In [ ]:
y_prob = model.predict_proba(X_test)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.show()

print("AUC:", round(roc_auc,3))

## K-Means Clustering (k=3)

Used for physiological grouping and risk stratification.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
df["Cluster"] = kmeans.fit_predict(X)

sns.scatterplot(
    x=df["HeartRate"],
    y=df["SpO2"],
    hue=df["Cluster"],
    palette="Set2"
)
plt.title("K-Means Clustering")
plt.show()

## Observations

- High Precision → Minimal False Alarms
- Moderate Recall → Some high-risk cases missed
- Very Low Execution Time → Near Real-Time Feasibility
- Clustering reveals distinct physiological patterns

The framework demonstrates scalable and reliable healthcare analytics
for distributed monitoring environments.